# Query `covid_country_stats`

This notebook demonstrates a paginated table scan, a partition-key query, and conversion of DynamoDB items into pandas DataFrames. It uses the standard boto3 credential chain; do not place credentials in the notebook.

In [ ]:
import os
from decimal import Decimal

import boto3
import pandas as pd
from boto3.dynamodb.conditions import Key

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
TABLE_NAME = os.getenv("TABLE_NAME", "covid_country_stats")

dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)
table = dynamodb.Table(TABLE_NAME)
TABLE_NAME

## Paginated `scan()`

A scan reads the current table snapshot. Pagination is required because one response is limited to 1 MB.

In [ ]:
items = []
scan_kwargs = {}

while True:
    response = table.scan(**scan_kwargs)
    items.extend(response.get("Items", []))
    last_key = response.get("LastEvaluatedKey")
    if not last_key:
        break
    scan_kwargs["ExclusiveStartKey"] = last_key

len(items)

In [ ]:
def make_dataframe(records):
    """Convert DynamoDB Decimals into display-friendly Python numbers."""
    def convert_value(value):
        if isinstance(value, Decimal):
            return int(value) if value == value.to_integral_value() else float(value)
        return value

    frame = pd.DataFrame(records)
    return frame.map(convert_value) if not frame.empty else frame

scan_df = make_dataframe(items)
scan_df.sort_values("active_cases", ascending=False).head(10)

## Partition-key `query()`

The table has only a partition key, so this query returns zero or one item. `GetItem` is preferable for a production single-country lookup, but `query()` is shown here as requested.

In [ ]:
country_name = "INDIA"
query_response = table.query(
    KeyConditionExpression=Key("country").eq(country_name)
)
query_df = make_dataframe(query_response.get("Items", []))
query_df